# Vigil-AWS: Strands Decider 2B Interactive Walkthrough
### Autonomous Cloud Incident & Remediation Commander

[![Hackathon](https://img.shields.io/badge/AWS%20Hackathon-Agent%20Experience-FF9900?logo=amazon-aws&logoColor=white)](https://builder.aws.com/build/hackathons/4f50957c-1d08-37ab-b11c-97516b9adeb4/agent-experience-hackathon-strands-decider)
[![Model](https://img.shields.io/badge/Model-Strands%20Decider%202B-10B981)](https://strandsagents.com/blog/introducing-strands-decider/)
[![Pattern](https://img.shields.io/badge/Pattern-%23guardrails-blue)](#)
[![Pattern](https://img.shields.io/badge/Pattern-%23model--routing-indigo)](#)

This interactive notebook demonstrates how **Strands Decider 2B** acts as a sub-115ms System-1 reflex guard for **Amazon Bedrock** agents in high-stakes AWS operational workflows.

## 1. Setup & Package Imports
We import the reusable `decider_guard` package and typed question factories.

In [1]:
import asyncio
import sys
from pathlib import Path

# Add workspace root to sys.path
ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from reusable_patterns.decider_guard import DeciderGroundedGuardrail, StrandsDeciderClient
from reusable_patterns.decider_guard.questions import (
    make_grounding_question,
    make_blast_radius_question,
    make_injection_check_question,
)
from reusable_patterns.decider_guard.router import DeciderModelRouter
from src.chaos_simulator import GameDayChaosEngine, ALL_SCENARIOS

client = StrandsDeciderClient(enable_fallback_emulator=True)
guardrail = DeciderGroundedGuardrail(client=client, confidence_threshold=0.85)
engine = GameDayChaosEngine(guardrail=guardrail)
print("Vigil-AWS System-1 Guardrail & Chaos Engine Ready!")

## 2. Scenario 1: Safe Operational Remediation (PROCEED)
**Incident:** Aurora RDS cluster memory exhaustion during traffic surge. Legitimate rolling reboot requested.
**Expected Action:** `PROCEED` in sub-120ms.

In [2]:
res1 = await engine.run_scenario(ALL_SCENARIOS[0])
print(f"Action Taken:   {res1['action_taken']}")
print(f"Reflex Latency: {res1['latency_ms']:.1f} ms")
print(f"Grounding Score:{res1['grounding_score']:.2f}")
print(f"Reasoning:      {res1['reasoning']}")

## 3. Scenario 2: Adversarial Log Injection Attack (DENY)
**Incident:** Error log injects instruction to delete `AdminAccessRole` IAM role.
**Expected Action:** `DENY` hard-stop intervention.

In [3]:
res2 = await engine.run_scenario(ALL_SCENARIOS[1])
print(f"Action Taken:   {res2['action_taken']}")
print(f"Reflex Latency: {res2['latency_ms']:.1f} ms")
print(f"Grounding Score:{res2['grounding_score']:.2f}")
print(f"Reasoning:      {res2['reasoning']}")

## 4. Scenario 3: High Blast-Radius Termination (CONFIRM)
**Incident:** Request to terminate 4 production EC2 instances during peak hours without maintenance window tag.
**Expected Action:** `CONFIRM` (Human-in-the-Loop Webhook).

In [4]:
res3 = await engine.run_scenario(ALL_SCENARIOS[2])
print(f"Action Taken:   {res3['action_taken']}")
print(f"Reflex Latency: {res3['latency_ms']:.1f} ms")
print(f"Grounding Score:{res3['grounding_score']:.2f}")
print(f"Reasoning:      {res3['reasoning']}")

## 5. Scenario 4: Ambiguous S3 Bucket Policy (GUIDE)
**Incident:** Prompt asks to 'clear bucket policy' without naming specific bucket ARN.
**Expected Action:** `GUIDE` (Self-healing feedback loop).

In [5]:
res4 = await engine.run_scenario(ALL_SCENARIOS[3])
print(f"Action Taken:   {res4['action_taken']}")
print(f"Reflex Latency: {res4['latency_ms']:.1f} ms")
print(f"Grounding Score:{res4['grounding_score']:.2f}")
print(f"Reasoning:      {res4['reasoning']}")

## 6. Dynamic Model Routing (#model-routing)
Decider 2B routes simple alerts to Bedrock Haiku (80% cost savings) and complex cascading outages to Bedrock Claude 3.5 Sonnet.

In [6]:
router = DeciderModelRouter(client=client)
r_simple = await router.route_incident("Query p99 latency for checkout API over past 10m")
print("Simple Query -> Model:", r_simple['selected_model'], "| Tier:", r_simple['model_tier'])

r_complex = await router.route_incident("Cascading multi-tier deadlock across Aurora writer and DynamoDB billing partition")
print("Complex Outage -> Model:", r_complex['selected_model'], "| Tier:", r_complex['model_tier'])

## 7. CloudWatch Embedded Metric Format (EMF)
Emitting structured telemetry without sidecar agents.

In [7]:
from reusable_patterns.decider_guard.calibration import export_cloudwatch_emf
import json

emf_payload = export_cloudwatch_emf(
    metric_namespace="VigilAWS/Decider2B",
    dimensions={"Environment": "Production", "Action": "Proceed"},
    metrics={"ReflexLatencyMs": 108.4, "GroundingScore": 0.94, "CostSavedUSD": 0.0042}
)
print(json.dumps(emf_payload, indent=2))